<h1 id="AMSDN:-Adaptive-Multi-Scale-Defense-Network">AMSDN: Adaptive Multi-Scale Defense Network<a class="anchor-link" href="#AMSDN:-Adaptive-Multi-Scale-Defense-Network">¶</a></h1>


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/AMSDN


/content/drive/MyDrive/AMSDN


<h2 id="1.-Setup-and-Installation">1. Setup and Installation<a class="anchor-link" href="#1.-Setup-and-Installation">¶</a></h2>


In [ ]:
!pip install -q torch torchvision timm matplotlib scikit-learn tqdm tensorboard scipy


In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


GPU Available: True
GPU Name: Tesla T4


<h2 id="2.-Quick-Model-Test">2. Quick Model Test<a class="anchor-link" href="#2.-Quick-Model-Test">¶</a></h2>


In [ ]:
from models.amsdn import AMSDN
from data.cifar10 import CIFAR10DataModule
model = AMSDN(num_classes=10, pretrained=False)
print(f"✓ Model created successfully")


x = torch.randn(2, 3, 32, 32)
outputs = model(x, return_detailed=True)
print(f"✓ Forward pass successful")
print(f"  Logits shape: {outputs['logits'].shape}")
print(f"  Anomaly detection: {outputs['is_adversarial']}")


✓ Model created successfully
✓ Forward pass successful
  Logits shape: torch.Size([2, 10])
  Anomaly detection: tensor([False, False])


<h2 id="3.-Stage-1:-SSRT-Pretraining-(Optional)">3. Stage 1: SSRT Pretraining (Optional)<a class="anchor-link" href="#3.-Stage-1:-SSRT-Pretraining-(Optional)">¶</a></h2>


In [ ]:
!python training/pretrain_ssrt.py


2026-03-27 05:33:38.975547: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774589619.011147   12934 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774589619.022347   12934 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774589619.050894   12934 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774589619.050925   12934 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774589619.050933   12934 computation_placer.cc:177] computation placer alr

<h2 id="4.-Stage-2:-Adversarial-Training">4. Stage 2: Adversarial Training<a class="anchor-link" href="#4.-Stage-2:-Adversarial-Training">¶</a></h2>


In [ ]:
!python training/adversarial_train.py


2026-03-27 06:54:58.055506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774594498.077179   34225 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774594498.085697   34225 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774594498.110931   34225 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774594498.110962   34225 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774594498.110966   34225 computation_placer.cc:177] computation placer alr

In [ ]:
!ls ./checkpoints/adversarial


amsdn_best.pth	amsdn_last.pth	logs


<h2 id="5.-Stage-3:-Fast-Robust-Tune">5. Stage 3: Fast Robust Tune<a class="anchor-link" href="#5.-Stage-3:-Fast-Robust-Tune">¶</a></h2>


In [ ]:
!python training/fast_robust_tune.py


2026-03-27 07:46:17.406802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774597577.441805   47574 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774597577.453219   47574 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774597577.480852   47574 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774597577.480892   47574 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774597577.480899   47574 computation_placer.cc:177] computation placer alr

<h2 id="6.-Evaluation">6. Evaluation<a class="anchor-link" href="#6.-Evaluation">¶</a></h2>


In [ ]:
!python evaluation/evaluate.py

import json
with open('./results/evaluation_results.json', 'r') as f:
    results = json.load(f)

print("\n=== EVALUATION RESULTS ===")
for attack, metrics in results.items():
    print(f"\n{attack}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.2f}")


2026-03-27 08:06:33.145595: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774598793.167240   53071 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774598793.174434   53071 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774598793.192540   53071 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774598793.192565   53071 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774598793.192568   53071 computation_placer.cc:177] computation placer alr

<h2 id="7.-Visualization">7. Visualization<a class="anchor-link" href="#7.-Visualization">¶</a></h2>


In [ ]:
# Visualize adversarial examples
from utils.helpers import visualize_adversarial_examples
from attacks.patch_attacks import AdversarialPatch
from data.cifar10 import CIFAR10DataModule
import torch

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AMSDN(num_classes=10, pretrained=False).to(device)
checkpoint = torch.load('./checkpoints/fast_robust/amsdn_fast_robust_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get test images
data_module = CIFAR10DataModule(batch_size=8)
_, test_loader = data_module.get_loaders()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

# Generate adversarial examples
patch_attack = AdversarialPatch(patch_size=4, epsilon=0.3)
adv_images = patch_attack.apply(images, model, labels, optimize=True)

# Predict
with torch.no_grad():
    preds_clean = model(images).argmax(dim=1)
    preds_adv = model(adv_images).argmax(dim=1)

# Visualize
visualize_adversarial_examples(
    images, adv_images, labels, preds_clean, preds_adv,
    save_path='./adversarial_examples.png'
)

from IPython.display import Image, display
display(Image('./adversarial_examples.png'))


Saved visualization to ./adversarial_examples.png


In [ ]:
# Visualize adversarial examples
from IPython.display import Image, display
import os
import torch

from utils.helpers import visualize_adversarial_examples
from attacks.patch_attacks import AdversarialPatch
from data.cifar10 import CIFAR10DataModule
from models.amsdn import AMSDN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
model = AMSDN(num_classes=10, pretrained=False).to(device)

checkpoint_candidates = [
    './checkpoints/fast_robust_low_resource/amsdn_fast_robust_best.pth',
    './checkpoints/fast_robust/amsdn_fast_robust_best.pth',
    './checkpoints/adversarial/amsdn_best.pth',
]

checkpoint_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
if checkpoint_path is None:
    raise FileNotFoundError("No trained checkpoint found.")

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get test images
data_module = CIFAR10DataModule(batch_size=8)
_, test_loader = data_module.get_loaders()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

# Generate adversarial examples
patch_attack = AdversarialPatch(patch_size=4, epsilon=0.3)
adv_images = patch_attack.apply(images, model, labels, optimize=True)

# Predict
with torch.no_grad():
    preds_clean = model(images).argmax(dim=1)
    preds_adv = model(adv_images).argmax(dim=1)

# Visualize
visualize_adversarial_examples(
    clean_images=images.detach().cpu(),
    adv_images=adv_images.detach().cpu(),
    labels=labels.detach().cpu(),
    preds_clean=preds_clean.detach().cpu(),
    preds_adv=preds_adv.detach().cpu(),
    save_path='./adversarial_examples.png'
)

display(Image('./adversarial_examples.png'))


Saved visualization to ./adversarial_examples.png


In [ ]:
from IPython.display import Image, display
import os
import torch

from utils.helpers import visualize_adversarial_examples
from attacks.pixel_attacks import FewPixelAttack
from data.cifar10 import CIFAR10DataModule
from models.amsdn import AMSDN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AMSDN(num_classes=10, pretrained=False).to(device)

checkpoint_candidates = [
    './checkpoints/fast_robust_low_resource/amsdn_fast_robust_best.pth',
    './checkpoints/fast_robust/amsdn_fast_robust_best.pth',
    './checkpoints/adversarial/amsdn_best.pth',
]

checkpoint_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
if checkpoint_path is None:
    raise FileNotFoundError("No trained checkpoint found.")

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

data_module = CIFAR10DataModule(batch_size=8)
_, test_loader = data_module.get_loaders()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

pixel_attack = FewPixelAttack(num_pixels=5, epsilon=0.5, num_iterations=30)
adv_images = pixel_attack.attack(images, model, labels)

with torch.no_grad():
    preds_clean = model(images).argmax(dim=1)
    preds_adv = model(adv_images).argmax(dim=1)

visualize_adversarial_examples(
    clean_images=images.detach().cpu(),
    adv_images=adv_images.detach().cpu(),
    labels=labels.detach().cpu(),
    preds_clean=preds_clean.detach().cpu(),
    preds_adv=preds_adv.detach().cpu(),
    save_path='./pixel_attack_examples.png'
)

display(Image('./pixel_attack_examples.png'))


Saved visualization to ./pixel_attack_examples.png


In [ ]:
from IPython.display import Image, display
import os
import torch

from utils.helpers import visualize_adversarial_examples
from training.adversarial_train import PGDAttack
from data.cifar10 import CIFAR10DataModule
from models.amsdn import AMSDN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AMSDN(num_classes=10, pretrained=False).to(device)

checkpoint_candidates = [
    './checkpoints/fast_robust_low_resource/amsdn_fast_robust_best.pth',
    './checkpoints/fast_robust/amsdn_fast_robust_best.pth',
    './checkpoints/adversarial/amsdn_best.pth',
]

checkpoint_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
if checkpoint_path is None:
    raise FileNotFoundError("No trained checkpoint found.")

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

data_module = CIFAR10DataModule(batch_size=8)
_, test_loader = data_module.get_loaders()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

pgd_attack = PGDAttack(epsilon=8/255, alpha=2/255, num_steps=20)
adv_images = pgd_attack.generate(model, images, labels)

with torch.no_grad():
    preds_clean = model(images).argmax(dim=1)
    preds_adv = model(adv_images).argmax(dim=1)

visualize_adversarial_examples(
    clean_images=images.detach().cpu(),
    adv_images=adv_images.detach().cpu(),
    labels=labels.detach().cpu(),
    preds_clean=preds_clean.detach().cpu(),
    preds_adv=preds_adv.detach().cpu(),
    save_path='./pgd_attack_examples.png'
)

display(Image('./pgd_attack_examples.png'))


Saved visualization to ./pgd_attack_examples.png


In [ ]:
from IPython.display import Image, display
import os
import torch

from utils.helpers import visualize_adversarial_examples
from attacks.patch_attacks import AdversarialPatch
from data.cifar10 import CIFAR10DataModule
from models.amsdn import AMSDN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AMSDN(num_classes=10, pretrained=False).to(device)

checkpoint_candidates = [
    './checkpoints/fast_robust_low_resource/amsdn_fast_robust_best.pth',
    './checkpoints/fast_robust/amsdn_fast_robust_best.pth',
    './checkpoints/adversarial/amsdn_best.pth',
]

checkpoint_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
if checkpoint_path is None:
    raise FileNotFoundError("No trained checkpoint found.")

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

data_module = CIFAR10DataModule(batch_size=8)
_, test_loader = data_module.get_loaders()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

patch_attack = AdversarialPatch(patch_size=4, epsilon=0.3)
adv_images = patch_attack.apply(images, model, labels, optimize=True)

with torch.no_grad():
    preds_clean = model(images).argmax(dim=1)
    preds_adv = model(adv_images).argmax(dim=1)

visualize_adversarial_examples(
    clean_images=images.detach().cpu(),
    adv_images=adv_images.detach().cpu(),
    labels=labels.detach().cpu(),
    preds_clean=preds_clean.detach().cpu(),
    preds_adv=preds_adv.detach().cpu(),
    save_path='./patch_attack_examples.png'
)

display(Image('./patch_attack_examples.png'))


Saved visualization to ./patch_attack_examples.png


<h2 id="9.-Download-Results">9. Download Results<a class="anchor-link" href="#9.-Download-Results">¶</a></h2>


In [ ]:
!zip -r amsdn_results.zip checkpoints/ results/ *.png

from google.colab import files
files.download('amsdn_results.zip')


  adding: checkpoints/ (stored 0%)
  adding: checkpoints/ssrt/ (stored 0%)
  adding: checkpoints/ssrt/logs/ (stored 0%)
  adding: checkpoints/ssrt/logs/events.out.tfevents.1774589640.3b1e12907940.12934.0 (deflated 71%)
  adding: checkpoints/ssrt/ssrt_best.pth (deflated 12%)
  adding: checkpoints/ssrt/ssrt_epoch_10.pth (deflated 12%)
  adding: checkpoints/adversarial/ (stored 0%)
  adding: checkpoints/adversarial/logs/ (stored 0%)
  adding: checkpoints/adversarial/logs/events.out.tfevents.1774594513.3b1e12907940.34225.0 (deflated 9%)
  adding: checkpoints/adversarial/amsdn_best.pth (deflated 12%)
  adding: checkpoints/adversarial/amsdn_last.pth (deflated 12%)
  adding: checkpoints/fast_robust/ (stored 0%)
  adding: checkpoints/fast_robust/amsdn_fast_robust_best.pth (deflated 8%)
  adding: results/ (stored 0%)
  adding: results/evaluation_results.json (deflated 72%)
  adding: adversarial_examples.png (deflated 13%)


<h2 id="10.-Quick-Demo-(No-Training)">10. Quick Demo (No Training)<a class="anchor-link" href="#10.-Quick-Demo-(No-Training)">¶</a></h2><p>Run this cell to test AMSDN without training (uses random weights)</p>


In [ ]:
print("=== AMSDN QUICK DEMO ===")
print("Testing pipeline with random weights...\n")

from models.amsdn import AMSDN
from data.cifar10 import CIFAR10DataModule
from attacks.patch_attacks import AdversarialPatch
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model
model = AMSDN(num_classes=10, pretrained=False).to(device)
model.eval()
print("✓ Model initialized")

# Data
data_module = CIFAR10DataModule(batch_size=4)
_, test_loader = data_module.get_loaders()
images, labels = next(iter(test_loader))
images = images.to(device)
print("✓ Data loaded")

# Clean prediction
with torch.no_grad():
    outputs_clean = model(images, return_detailed=True)
    print(f"\n Clean predictions: {outputs_clean['logits'].argmax(dim=1)}")
    print(f" True labels: {labels}")
    print(f" Anomaly scores: {outputs_clean['avg_anomaly_score']}")
    print(f" Detected as adversarial: {outputs_clean['is_adversarial']}")

# Attack
attack = AdversarialPatch(patch_size=4, epsilon=0.3, num_steps=50)
adv_images = attack.apply(images, model, labels, optimize=False)  # Random patch
print("\n✓ Generated adversarial examples")

# Adversarial prediction
with torch.no_grad():
    outputs_adv = model(adv_images, return_detailed=True)
    print(f"\n Adversarial predictions: {outputs_adv['logits'].argmax(dim=1)}")
    print(f" Anomaly scores: {outputs_adv['avg_anomaly_score']}")
    print(f" Detected as adversarial: {outputs_adv['is_adversarial']}")

print("\n=== DEMO COMPLETE ===")
print("Note: Model uses random weights. Train for real performance.")


=== AMSDN QUICK DEMO ===
Testing pipeline with random weights...

✓ Model initialized
✓ Data loaded

 Clean predictions: tensor([4, 4, 8, 2], device='cuda:0')
 True labels: tensor([3, 8, 8, 0])
 Anomaly scores: tensor([0.4963, 0.4961, 0.4950, 0.4961], device='cuda:0')
 Detected as adversarial: tensor([False, False, False, False], device='cuda:0')

✓ Generated adversarial examples

 Adversarial predictions: tensor([4, 4, 8, 2], device='cuda:0')
 Anomaly scores: tensor([0.4961, 0.4959, 0.4950, 0.4968], device='cuda:0')
 Detected as adversarial: tensor([False, False, False, False], device='cuda:0')

=== DEMO COMPLETE ===
Note: Model uses random weights. Train for real performance.
